In [ ]:
import os
import shutil
from time import sleep
import time
from linecache import getline
from IPython.display import clear_output

import sys
sys.path.insert(0, 'name_of_folder')
from config import FOLDER_NAME, SIF_FILE, NAME_OF_FOLDER

class File:
    """
    A SLURM output file representation, containing:
    - file_name
    - slurm job name
    - name_for_user (arg-name)
    - last_line (progress line)
    """

    def __init__(self, file_name):
        """
        Initialize file metadata and extract useful information.
        """
        self.delete_me = False
        self.file_name = file_name

        file_root, suffix = file_name.split('.')
        if suffix != 'out':
            self.delete_me = True
            return

        self.slurm_name = file_root.split('_')[0]
        self.name_for_user = 'Starting...'

        lines = open(self.file_name).readlines()

        # Find name for user
        for i, line in enumerate(lines):
            if line[:5] == 'name:' and lines[i + 1][0] != '\t':
                self.name_for_user = lines[i + 1].rstrip('\n')
                break

        # Find last non-empty line
        self.last_line = 'Starting...'
        if len(lines) > 0:
            last = -1
            while last >= -len(lines) and lines[last] == '\n':
                last -= 1
            if abs(last) <= len(lines):
                self.last_line = lines[last]

        self.last_trim_time = time.time()

    def done(self):
        """
        Return True if this SLURM file reports completion.
        """
        return self.last_line.endswith('Done!\n')

    def __lt__(self, other):
        """
        Sort files by SLURM name.
        """
        return self.slurm_name < other.slurm_name


# -------------------------------------------------------------
# Main monitoring loop
# -------------------------------------------------------------
first = True

while True:
    file_names = [f for f in os.listdir() if f.startswith('slurm')]

    files = []
    for name in file_names:
        f = File(name)
        if not f.delete_me:
            files.append(f)

    files.sort()

    ongoing = []
    finished = []

    for f in files:
        if f.done():
            finished.append(f)
        else:
            ongoing.append(f)

    # Wait cycle before refresh (after first pass)
    if not first:
        print('\n\n')
        for i in range(10):
            print(i + 1, end='... ')
            sleep(1)

    first = False

    clear_output(wait=True)

    # Determine alignment width
    uniform_length = 0
    for f in ongoing + finished:
        if f.last_line != '\n':
            label = f'{f.name_for_user} ({f.file_name}):'
            uniform_length = max(uniform_length, len(label))

    # -------------------------------------------------------------
    # Print Ongoing
    # -------------------------------------------------------------
    print('Ongoing:')
    if len(ongoing) == 0:
        print('None.')
    else:
        for f in ongoing:
            label = f'{f.name_for_user} ({f.file_name}):'
            spacing = ' ' * (uniform_length - len(label))
            last_line = f.last_line.rstrip('\n')
            print(f'{label}{spacing}\t{last_line}')

    # -------------------------------------------------------------
    # Print Finished
    # -------------------------------------------------------------
    print('\n\n\nFinished:')
    if len(finished) == 0:
        print('None.')
    else:
        for f in finished:
            label = f'{f.name_for_user} ({f.file_name}):'
            spacing = ' ' * (uniform_length - len(label))
            last_line = f.last_line.rstrip('\n')
            print(f'{label}{spacing}\t{last_line}')

    # Names for display or chaining
    finished_names = [f.name_for_user for f in finished]
    ongoing_names = [f.name_for_user for f in ongoing]


Ongoing:
ef_q2 (slurm-19431860.out):	0:02:17.514322 (1 day, 22:44:06.485678 left):	0   0   0   0   0   0   0   0   0   0   ##.
ef_q2 (slurm-19431861.out):	0:02:17.647860 (1 day, 20:04:46.352140 left):	0   0   0   0   0   0   0   0   0   0   ##.
ef_q2 (slurm-19431862.out):	0:02:17.999155 (1 day, 21:57:41.000845 left):	0   0   0   0   0   0   0   0   0   0   ##.
ef_q2 (slurm-19431863.out):	0:01:47.275393 (2 days, 1:38:04.724607 left):	0   0   0   0   0   0   0   0   0   0   ##.
ef_q2 (slurm-19431864.out):	0:02:17.694237 (1 day, 20:05:39.305763 left):	0   0   0   0   0   0   0   0   0   0   ##.
ef_q2 (slurm-19431866.out):	0:02:17.099469 (1 day, 19:54:13.900531 left):	0   0   0   0   0   0   0   0   0   0   ##.
ef_q2 (slurm-19431867.out):	0:02:17.561471 (1 day, 16:11:03.438529 left):	0   0   0   0   0   0   0   0   0   0   ##.
ef_q2 (slurm-19431868.out):	0:02:17.119453 (1 day, 16:03:18.880547 left):	0   0   0   0   0   0   0   0   0   0   ##.
ef_q2 (slurm-19431869.out):	0:02:17.629487 (1 d

In [4]:
from datetime import timedelta
import re


# Threshold for job cancellation
threshold = timedelta(days=2, hours=0)


def parse_time_left(line):
    """
    Extracts a timedelta from a line containing (X days, HH:MM:SS left)
    """
    match = re.search(r'\((\d+) day[s]?, (\d+):(\d+):([\d.]+) left\)', line)
    if not match:
        return None
    days, hours, minutes, seconds = map(float, match.groups())
    return timedelta(days=days, hours=hours, minutes=minutes, seconds=seconds)


# Lists for confirmed cancellations
confirmed_args = []
job_ids = []
slurm_files = []

# Prompt user to confirm cancelable jobs
for file in ongoing:
    time_left = parse_time_left(file.last_line)
    if time_left and time_left > threshold:
        arg_name = file.name_for_user.strip()
        job_file = file.file_name
        job_number = job_file.replace('slurm-', '').replace('.out', '')

        confirm = input(
            f'Cancel job \'{arg_name}\' (Job ID {job_number}) and delete \'{job_file}\'? (y/n): '
        ).strip().lower()

        if confirm == 'y':
            confirmed_args.append(arg_name)
            job_ids.append(job_number)
            slurm_files.append(job_file)
            print(f'✅ Marked for cancel: {job_number}, file: {job_file}')
        else:
            print(f'⏭️ Skipped: {job_file}')


# Output suggested bash commands
if confirmed_args:
    arg_list_str = ' '.join(confirmed_args)
    job_ids_str = ' '.join(job_ids)
    files_str = ' '.join(slurm_files)

    print('\n\nSuggested bash commands for the cluster:\n')
    print(f'scancel {job_ids_str}')
    print(f'rm {files_str}')
    print('\nSuggested bash command to rerun:')
    print(f'bash NAME_OF_FOLDER/bash/NAME_OF_FOLDER.sh "arg_list=({arg_list_str})" "comp=deigo" "agents=10"')
else:
    print(f'\n\nNo jobs selected for cancel/deletion over threshold of {threshold}.')




No jobs selected for cancel/deletion over threshold of 2 days, 0:00:00.


In [5]:
arg_title_func = 'finish_dicts'
#arg_title_func = 'plotting'

if len(finished_names) > 0:
    print('\n\nFINISHED Generated arg_title:')
    print(
        f'singularity exec NAME_OF_FOLDER.sif python NAME_OF_FOLDER/{arg_title_func}.py '
        f'--comp deigo --arg_title ___{"+".join(finished_names)}___ --arg_name {arg_title_func}'
    )
    print()
    for finished_name in finished_names:
        print(
            f'singularity exec NAME_OF_FOLDER.sif python NAME_OF_FOLDER/{arg_title_func}.py '
            f'--comp deigo --arg_title ___{finished_name}___ --arg_name {arg_title_func}'
        )
    print('\n\nFINISHED Generated arg_list:')
    print(
        f'bash NAME_OF_FOLDER/bash/NAME_OF_FOLDER.sh \'arg_list=({" ".join(finished_names)})\' '
        f'\'comp=deigo\' \'agents=0\''
    )
    print()
    for finished_name in finished_names:
        print(
            f'bash NAME_OF_FOLDER/bash/NAME_OF_FOLDER.sh \'arg_list=({finished_name})\' '
            f'\'comp=deigo\' \'agents=0\''
        )


print('\n\nONGOING Generated arg_title:')
print(
    f'singularity exec NAME_OF_FOLDER.sif python NAME_OF_FOLDER/{arg_title_func}.py '
    f'--comp deigo --arg_title ___{"+".join(ongoing_names)}___ --arg_name {arg_title_func} --temp True'
)
print()
for ongoing_name in ongoing_names:
    print(
        f'singularity exec NAME_OF_FOLDER.sif python NAME_OF_FOLDER/{arg_title_func}.py '
        f'--comp deigo --arg_title ___{ongoing_name}___ --arg_name {arg_title_func} --temp True'
    )

print('\n\nONGOING Generated arg_list:')
print(
    f'bash NAME_OF_FOLDER/bash/NAME_OF_FOLDER.sh \'arg_list=({" ".join(ongoing_names)})\' '
    f'\'comp=deigo\' \'agents=0\''
)
print()
for ongoing_name in ongoing_names:
    print(
        f'bash NAME_OF_FOLDER/bash/NAME_OF_FOLDER.sh \'arg_list=({ongoing_name})\' '
        f'\'comp=deigo\' \'agents=0\''
    )


print('\n\nALL Generated arg_title:')
combined = finished_names + ongoing_names
print(
    f'singularity exec NAME_OF_FOLDER.sif python NAME_OF_FOLDER/{arg_title_func}.py '
    f'--comp deigo --arg_title ___{"+".join(combined)}___ --arg_name {arg_title_func} --temp True'
)
print()
for name in combined:
    print(
        f'singularity exec NAME_OF_FOLDER.sif python NAME_OF_FOLDER/{arg_title_func}.py '
        f'--comp deigo --arg_title ___{name}___ --arg_name {arg_title_func} --temp True'
    )

print('\n\nALL Generated arg_list:')
print(
    f'bash NAME_OF_FOLDER/bash/NAME_OF_FOLDER.sh \'arg_list=({" ".join(combined)})\' '
    f'\'comp=deigo\' \'agents=0\''
)
print()
for name in combined:
    print(
        f'bash NAME_OF_FOLDER/bash/NAME_OF_FOLDER.sh \'arg_list=({name})\' '
        f'\'comp=deigo\' \'agents=0\''
    )




ONGOING Generated arg_title:
singularity exec communication.sif python communication/plotting.py --comp deigo --arg_title ___ef_q2___ --arg_name plotting --temp True

singularity exec communication.sif python communication/plotting.py --comp deigo --arg_title ___ef_q2___ --arg_name plotting --temp True


ONGOING Generated arg_list:
bash communication/bash/communication.sh 'arg_list=(ef_q2' 'comp=deigo' 'agents=0'

bash communication/bash/communication.sh 'arg_list=(ef_q2' 'comp=deigo' 'agents=0'


ALL Generated arg_title:
singularity exec communication.sif python communication/plotting.py --comp deigo --arg_title ___+ef_q2___ --arg_name plotting --temp True

singularity exec communication.sif python communication/plotting.py --comp deigo --arg_title ___ef_q2___ --arg_name plotting --temp True


ALL Generated arg_list:
bash communication/bash/communication.sh 'arg_list=( ef_q2' 'comp=deigo' 'agents=0'

bash communication/bash/communication.sh 'arg_list=(ef_q2' 'comp=deigo' 'agents=0'
